In [1]:
import time
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("nyc311-read-parquet")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 11:39:01 WARN Utils: Your hostname, MacBook-Pro-23.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.83 instead (on interface en0)
26/10/13 11:39:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 11:39:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/13 11:39:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
PARQUET_PATH = Path.cwd() / "data" / "311-service-requests-from-2010-to-present.parquet"

start = time.perf_counter()
df = spark.read.parquet(str(PARQUET_PATH))
print(f"Read Parquet in {time.perf_counter() - start:.2f}s")

df.printSchema()

Read Parquet in 1.00s
root
 |-- Unique Key: integer (nullable = true)
 |-- Created Date: timestamp (nullable = true)
 |-- Closed Date: timestamp (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Agency Name: string (nullable = true)
 |-- Complaint Type: string (nullable = true)
 |-- Descriptor: string (nullable = true)
 |-- Location Type: string (nullable = true)
 |-- Incident Zip: integer (nullable = true)
 |-- Incident Address: string (nullable = true)
 |-- Street Name: string (nullable = true)
 |-- Cross Street 1: string (nullable = true)
 |-- Cross Street 2: string (nullable = true)
 |-- Intersection Street 1: string (nullable = true)
 |-- Intersection Street 2: string (nullable = true)
 |-- Address Type: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Landmark: string (nullable = true)
 |-- Facility Type: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Due Date: string (nullable = true)
 |-- Resolution Description: string (nullable 

Rename columns to snake_case

In [3]:
import re


def to_snake_case(name: str) -> str:
    return re.sub(r"[^0-9a-z]+", "_", name.strip().lower()).strip("_")


df_renamed = df
for old_name in df.columns:
    df_renamed = df_renamed.withColumnRenamed(old_name, to_snake_case(old_name))

assert len(set(df_renamed.columns)) == len(df.columns), "renaming produced duplicate column names"
df_renamed.printSchema()

root
 |-- unique_key: integer (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- closed_date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- incident_zip: integer (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- cross_street_1: string (nullable = true)
 |-- cross_street_2: string (nullable = true)
 |-- intersection_street_1: string (nullable = true)
 |-- intersection_street_2: string (nullable = true)
 |-- address_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- landmark: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- due_date: string (nullable = true)
 |-- resolution_description: string (nullable = true)
 |-- resolutio

Drop columns

Identify columns where every value is NULL

Drop unused columns

In [4]:
columns_to_drop = [
    "vehicle_type",
    "taxi_company_borough",
    "bridge_highway_name",
    "facility_type",
    "due_date",
]

missing = [c for c in columns_to_drop if c not in df_renamed.columns]
if missing:
    print(f"Not found, skipped: {missing}")

df_dropped = df_renamed.drop(*columns_to_drop)
print(f"Columns: {len(df_renamed.columns)} -> {len(df_dropped.columns)}")
df_dropped.printSchema()

Columns: 46 -> 41
root
 |-- unique_key: integer (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- closed_date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- incident_zip: integer (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- cross_street_1: string (nullable = true)
 |-- cross_street_2: string (nullable = true)
 |-- intersection_street_1: string (nullable = true)
 |-- intersection_street_2: string (nullable = true)
 |-- address_type: string (nullable = true)
 |-- city: string (nullable = true)
 |-- landmark: string (nullable = true)
 |-- status: string (nullable = true)
 |-- resolution_description: string (nullable = true)
 |-- resolution_action_updated_date: timestamp (nullable = true)
 |-- community_b

Derive resolution time, SLA flag, and created hour / day of week

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

SLA_HOURS = 72

df_features = (
    df_dropped
    .withColumn(
        "resolution_hours",
        (col("closed_date").cast("long") - col("created_date").cast("long")) / 3600,
    )
    # NULL for still-open requests (closed_date is NULL)
    .withColumn("closed_within_sla", col("resolution_hours") <= SLA_HOURS)  # on unrounded hours
    .withColumn("resolution_hours", F.round(col("resolution_hours"), 2))
    .withColumn("created_hour", F.hour(col("created_date")))
    .withColumn("created_day_of_week", F.dayofweek(col("created_date")))  # 1=Sunday ... 7=Saturday
    .withColumn("created_day_name", F.date_format(col("created_date"), "EEEE"))
)

df_features.select(
    "created_date", "closed_date", "resolution_hours", "closed_within_sla",
    "created_hour", "created_day_of_week", "created_day_name",
).where("closed_date IS NOT NULL AND closed_within_sla = false").show(10, truncate=False)

+-------------------+-------------------+----------------+-----------------+------------+-------------------+----------------+
|created_date       |closed_date        |resolution_hours|closed_within_sla|created_hour|created_day_of_week|created_day_name|
+-------------------+-------------------+----------------+-----------------+------------+-------------------+----------------+
|2019-11-27 11:54:00|2019-11-30 12:00:00|72.1            |false            |11          |4                  |Wednesday       |
|2019-11-27 11:10:00|2019-11-30 12:00:00|72.83           |false            |11          |4                  |Wednesday       |
|2019-11-27 10:59:00|2019-11-30 12:00:00|73.02           |false            |10          |4                  |Wednesday       |
|2019-11-27 10:35:00|2019-11-30 12:00:00|73.42           |false            |10          |4                  |Wednesday       |
|2019-11-27 10:17:00|2019-11-30 12:00:00|73.72           |false            |10          |4                  |We

Resolution hours with SQL expr() vs the column API

In [6]:
from pyspark.sql.functions import expr

# Same logic as df_features, written as SQL strings
df_features_sql = (
    df_dropped
    .withColumn(
        "resolution_hours",
        expr("(unix_timestamp(closed_date) - unix_timestamp(created_date)) / 3600"),
    )
    .withColumn("closed_within_sla", expr(f"resolution_hours <= {SLA_HOURS}"))
    .withColumn("resolution_hours", expr("round(resolution_hours, 2)"))
)

df_features_sql.select(
    "created_date", "closed_date", "resolution_hours", "closed_within_sla"
).where("closed_date IS NOT NULL AND closed_within_sla = false").show(10, truncate=False)

cols = ["unique_key", "resolution_hours", "closed_within_sla"]

# 2. Same physical plan?
api_plan = df_features.select(*cols)._jdf.queryExecution().executedPlan().toString()
sql_plan = df_features_sql.select(*cols)._jdf.queryExecution().executedPlan().toString()
print(f"Identical physical plans: {api_plan == sql_plan}")
df_features.select(*cols).explain()
df_features_sql.select(*cols).explain()

# 3. Timing (forces full evaluation without collecting rows)
for name, frame in [("column API", df_features), ("expr() SQL", df_features_sql)]:
    start = time.perf_counter()
    frame.select(*cols).write.format("noop").mode("overwrite").save()
    print(f"{name}: {time.perf_counter() - start:.2f}s")

+-------------------+-------------------+----------------+-----------------+
|created_date       |closed_date        |resolution_hours|closed_within_sla|
+-------------------+-------------------+----------------+-----------------+
|2019-11-27 11:54:00|2019-11-30 12:00:00|72.1            |false            |
|2019-11-27 11:10:00|2019-11-30 12:00:00|72.83           |false            |
|2019-11-27 10:59:00|2019-11-30 12:00:00|73.02           |false            |
|2019-11-27 10:35:00|2019-11-30 12:00:00|73.42           |false            |
|2019-11-27 10:17:00|2019-11-30 12:00:00|73.72           |false            |
|2019-11-27 10:02:00|2019-11-30 12:00:00|73.97           |false            |
|2019-11-27 09:38:00|2019-11-30 12:00:00|74.37           |false            |
|2019-11-27 09:27:00|2019-11-30 11:25:00|73.97           |false            |
|2019-11-27 08:48:00|2019-11-30 12:00:00|75.2            |false            |
|2019-11-27 08:40:00|2019-11-30 12:00:00|75.33           |false            |

column API: 2.61s


expr() SQL: 0.83s


Filter: closed requests in a given borough with a resolution description

In [7]:
BOROUGH = "BROOKLYN"  # change to filter another borough

df_filtered = df_features.filter(
    (col("status") == "Closed")
    & (F.upper(col("borough")) == BOROUGH.upper())
    & col("resolution_description").isNotNull()
)

print(f"{BOROUGH}: {df_filtered.count():,} of {df_features.count():,} rows")
df_filtered.select(
    "unique_key", "status", "borough", "resolution_description", "resolution_hours"
).show(10, truncate=60)

BROOKLYN: 5,817,135 of 21,960,000 rows
+----------+------+--------+------------------------------------------------------------+----------------+
|unique_key|status| borough|                                      resolution_description|resolution_hours|
+----------+------+--------+------------------------------------------------------------+----------------+
|  45052727|Closed|BROOKLYN|The Police Department responded to the complaint and with...|            0.92|
|  45054876|Closed|BROOKLYN|The Police Department responded and upon arrival those re...|            1.54|
|  45052486|Closed|BROOKLYN|The Police Department responded to the complaint but offi...|            0.96|
|  45054194|Closed|BROOKLYN|The Police Department responded to the complaint and dete...|            0.94|
|  45054197|Closed|BROOKLYN|The Police Department responded to the complaint and took...|            0.89|
|  45050528|Closed|BROOKLYN|The Police Department responded to the complaint but offi...|            1.82

Deduplicate: dropDuplicates on unique_key vs distinct() on all columns

In [8]:
start = time.perf_counter()
total = df_features.count()
all_cols = df_features.distinct().count()

by_key = df_features.dropDuplicates(["unique_key"]).count()
print(f"Total rows:                         {total:,}")
print(f"dropDuplicates on unique_key:       {by_key:,}")
print(f"distinct on all_cols:               {all_cols:,}")
print(f"Rows removed by key dropDuplicates: {total - by_key:,}")
print(f"Rows removed by distinct:           {total - all_cols:,}")

print(f"Took {time.perf_counter() - start:.2f}s")

# Keys that appear more than once, and how many differ in other columns
dup_keys = (
    df_features.groupBy("unique_key").count().filter(col("count") > 1)
)
print(f"unique_key appearing more than once: {dup_keys.count():,}")

26/10/13 11:39:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total rows:                         21,960,000
dropDuplicates on unique_key:       209,999
distinct on all_cols:               209,999
Rows removed by key dropDuplicates: 21,750,001
Rows removed by distinct:           21,750,001
Took 10.05s


unique_key appearing more than once: 70,000


Slowest-resolved complaints

In [9]:
LIMIT = 20

start = time.perf_counter()
slowest = (
    df_features
    .orderBy(col("resolution_hours").desc_nulls_last())
    .limit(LIMIT)
    .select(
        "unique_key", "complaint_type", "borough", "agency",
        "created_date", "closed_date", "resolution_hours",
    )
)
slowest.show(truncate=False)
print(f"Took {time.perf_counter() - start:.2f}s")

+----------+-----------------------------+-------------+------+-------------------+-------------------+----------------+
|unique_key|complaint_type               |borough      |agency|created_date       |closed_date        |resolution_hours|
+----------+-----------------------------+-------------+------+-------------------+-------------------+----------------+
|43861537  |General Construction/Plumbing|STATEN ISLAND|DOB   |2019-09-22 13:33:28|2019-11-29 00:00:00|1618.44         |
|43861537  |General Construction/Plumbing|STATEN ISLAND|DOB   |2019-09-22 13:33:28|2019-11-29 00:00:00|1618.44         |
|43861537  |General Construction/Plumbing|STATEN ISLAND|DOB   |2019-09-22 13:33:28|2019-11-29 00:00:00|1618.44         |
|43861537  |General Construction/Plumbing|STATEN ISLAND|DOB   |2019-09-22 13:33:28|2019-11-29 00:00:00|1618.44         |
|43861537  |General Construction/Plumbing|STATEN ISLAND|DOB   |2019-09-22 13:33:28|2019-11-29 00:00:00|1618.44         |
|43861537  |General Construction

UDF: classify resolution outcome and extract the responding agency

In [10]:
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, StructField, StructType

# Ordered rules: first match wins, so specific phrases come before general ones
# (e.g. "no violations were issued" must beat "violations were issued").
OUTCOME_RULES = [
    ("DUPLICATE", r"\bduplicate\b"),
    ("OUT_OF_JURISDICTION", r"not fall under|jurisdiction|referred to|forwarded to|\bis out\b|out of (?:the|its)"),
    ("UNABLE_TO_ACCESS_OR_LOCATE", r"not able to gain|unable to gain|could not gain|gain (?:access|entry)|were gone|could not find|unable to locate|no one (?:was )?(?:home|available)"),
    ("NO_ACTION_NEEDED", r"no evid|no action|not necessary|no further action|no violations? (?:were )?(?:issued|found|observed)|did not observe a violation|found no (?:violation|condition)|not warranted|closed it"),
    ("PENDING_FOLLOW_UP", r"will (?:visit|inspect|investigate|perform|be addressed)|is being addressed|still open"),
    ("ACTION_TAKEN", r"took action|summons|violations? (?:were )?issued|issued a|repaired|removed|picked up|fixed|cleaned|resolved|corrected|abated|created a service request|performed the work"),
    ("INFO_PROVIDED", r"see notes|additional infor|status for this request|available at|available on|insufficient contact"),
]

# Known agency names; the free text varies too much for a generic "Department of ..." match
AGENCY_PATTERN = re.compile(
    r"\b(Police Department|Fire Department|Taxi and Limousine Commission"
    r"|Department of (?:Housing Preservation and Development|Environmental? Protection|Transportation"
    r"|Sanitation|Buildings|Parks and Recreation|Health and Mental Hygiene|Consumer Affairs"
    r"|Education|Finance|Homeless Services|Information Technology and Telecommunications))",
    re.IGNORECASE,
)

resolution_schema = StructType([
    StructField("resolution_outcome", StringType()),
    StructField("resolution_agency", StringType()),
])


def parse_resolution(text):
    if text is None:
        return None
    lowered = text.lower()
    outcome = next((label for label, pat in OUTCOME_RULES if re.search(pat, lowered)), "OTHER")
    match = AGENCY_PATTERN.search(text)
    agency = None
    if match:
        agency = match.group(1).replace("Environment Protection", "Environmental Protection")
        agency = re.sub(r"\s+", " ", agency)
    return (outcome, agency)


parse_resolution_udf = udf(parse_resolution, resolution_schema)

df_enriched = (
    df_features
    .withColumn("_parsed", parse_resolution_udf(col("resolution_description")))
    .withColumn("resolution_outcome", col("_parsed.resolution_outcome"))
    .withColumn("resolution_agency", col("_parsed.resolution_agency"))
    .drop("_parsed")
)

/Users/hs086u/Documents/learning/terraform-projects/nyc-311-data-pipeline/se-spark/.spark-env/lib/python3.14/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


In [11]:
# Inspect: outcome distribution, agency extraction, and what fell through to OTHER
start = time.perf_counter()
df_enriched.groupBy("resolution_outcome").count().orderBy(col("count").desc()).show(truncate=False)

df_enriched.filter(col("resolution_agency").isNotNull()) \
    .groupBy("resolution_agency").count().orderBy(col("count").desc()).show(15, truncate=False)

df_enriched.filter(col("resolution_outcome") == "OTHER") \
    .groupBy("resolution_description").count().orderBy(col("count").desc()).show(15, truncate=100)
print(f"Took {time.perf_counter() - start:.2f}s")

+--------------------------+-------+
|resolution_outcome        |count  |
+--------------------------+-------+
|NO_ACTION_NEEDED          |7566800|
|ACTION_TAKEN              |4045031|
|NULL                      |2613809|
|UNABLE_TO_ACCESS_OR_LOCATE|2466311|
|OTHER                     |2006712|
|INFO_PROVIDED             |1858052|
|PENDING_FOLLOW_UP         |624674 |
|DUPLICATE                 |458455 |
|OUT_OF_JURISDICTION       |320156 |
+--------------------------+-------+



+-----------------------------------------------------------+-------+
|resolution_agency                                          |count  |
+-----------------------------------------------------------+-------+
|Police Department                                          |9955694|
|Department of Transportation                               |2222498|
|Department of Housing Preservation and Development         |1588210|
|Department of Environmental Protection                     |1385862|
|Department of Buildings                                    |973011 |
|Department of Sanitation                                   |968303 |
|Department of Parks and Recreation                         |811574 |
|Department of Health and Mental Hygiene                    |175118 |
|Taxi and Limousine Commission                              |149945 |
|Department of Consumer Affairs                             |132987 |
|Department of Education                                    |13158  |
|Department of Homel

+----------------------------------------------------------------------------------------------------+------+
|                                                                              resolution_description| count|
+----------------------------------------------------------------------------------------------------+------+
|The Department of Transportation inspected the condition you reported and found that the condition m|139805|
|The Department of Transportation inspected and has requested the Department of Environmental Protect| 97842|
|The Department of Environmental Protection researched this complaint and determined that it could...| 93860|
|The Department of Health and Mental Hygiene has sent official written notification to the Owner/Land| 85538|
|"The Department of Buildings determined that the conditions described in this complaint were addr...| 85380|
|The Department of Housing Preservation and Development responded to a complaint of no heat or hot...| 64137|
|The Depar

26/10/13 13:44:14 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 609366 ms exceeds timeout 120000 ms
26/10/13 13:44:14 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/13 13:44:23 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora